# OpenAIRE Beginners Kit

The **OpenAIRE Graph** is an Open Access dataset containing metadata about research products (literature, datasets, software, and other research products) linked to other entities of the research ecosystem, such as organisations, grants, and data sources.

The large size of the OpenAIRE Graph is a major impediment for beginners to familiarise with the underlying data model and explore its contents. Working with the Graph in its full size typically requires access to a huge distributed computing infrastructure which cannot be easily accessible to everyone.

The OpenAIRE Beginner’s Kit aims to address this issue. It consists of two components: a subset of the Graph composed of the research products published between `2024-06-01` and `2024-12-31`, all the entities connected to them and the respective relationships, and the present Jupyter notebook that demonstrates how you can use `PySpark` to analyse the Graph and get answers to some interesting research questions.

This notebook is structured in sections to help you navigate the different cells; you can visualise the structure by clicking on the third icon in the leftmost menu.

You can run each cell individually (`shift+enter`with a cell highlighted) or run everything at once from the top menu.  

## Download the data

The OpenAIRE Beginners Kit data can be downloaded from [Zenodo record 14891799](https://zenodo.org/records/14891799). The following archives are used in this notebook:

- [communities_infrastructures.tar](https://zenodo.org/records/14891799/files/communities_infrastructures.tar?download=1)
- [dataset.tar](https://zenodo.org/records/14891799/files/dataset.tar?download=1)
- [datasource.tar](https://zenodo.org/records/14891799/files/datasource.tar?download=1)
- [organization.tar](https://zenodo.org/records/14891799/files/organization.tar?download=1)
- [otherresearchproduct.tar](https://zenodo.org/records/14891799/files/otherresearchproduct.tar?download=1)
- [project.tar](https://zenodo.org/records/14891799/files/project.tar?download=1)
- [publication.tar](https://zenodo.org/records/14891799/files/publication.tar?download=1)
- [relation.tar](https://zenodo.org/records/14891799/files/relation.tar?download=1)
- [software.tar](https://zenodo.org/records/14891799/files/software.tar?download=1)

Run the next cell to download the archives and extract them into `data/`.

In [28]:
import os
import tarfile

import urllib.request

record_url = "https://zenodo.org/records/14891799/files"
archives = [
    "communities_infrastructures",
    "dataset",
    "datasource",
    "organization",
    "otherresearchproduct",
    "project",
    "publication",
    "relation",
    "software",
]

os.makedirs("data", exist_ok=True)

for name in archives:
    tar_path = os.path.join("data", f"{name}.tar")
    if not os.path.exists(tar_path):
        print(f"Downloading {name}.tar ...")
        urllib.request.urlretrieve(f"{record_url}/{name}.tar?download=1", tar_path)
    print(f"Extracting {name}.tar ...")
    with tarfile.open(tar_path) as tar:
        tar.extractall("data")
    os.remove(tar_path)

print("Done.")

Extracting communities_infrastructures.tar ...


/var/folders/1z/xzf8b97x4jg374k9l_dn3glm0000gn/T/ipykernel_44880/101674813.py:28: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extractall("data")


Extracting dataset.tar ...
Extracting datasource.tar ...
Extracting organization.tar ...
Extracting otherresearchproduct.tar ...
Extracting project.tar ...
Extracting publication.tar ...
Extracting relation.tar ...
Extracting software.tar ...
Done.


# Import Libraries

In [2]:
import json
import glob
import pandas as pd
import re
import plotly.graph_objects as go
import pyspark.sql.functions as F
from pyspark.sql.functions import col
from pyspark.sql.types import StructType
from pyspark.sql import SparkSession
from IPython.display import JSON as pretty_print

import pandas as pd
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 100)
main_path = "data"


## Prepare data for analysis

The data subset is organised in part files under the `data/` folder. 


If you try to load the data straight into memory, one part file would fit

In [3]:
files = sorted(glob.glob(f"{main_path}/publication/part*"))
df = pd.read_json(files[0], compression='gzip', lines=True)
df.head(3)

,authors,openAccessColor,publiclyFunded,type,language,countries,subjects,mainTitle,subTitle,descriptions,publicationDate,publisher,embargoEndDate,sources,formats,contributors,coverages,bestAccessRight,container,documentationUrls,codeRepositoryUrl,programmingLanguage,contactPeople,contactGroups,tools,size,version,geoLocations,id,originalIds,pids,dateOfCollection,lastUpdateTimeStamp,indicators,instances,isGreen,isInDiamondJournal
0,"[{'fullName': 'Gobert, Thierry', 'name': 'Thierry', 'surname': 'Gobert', 'rank': 1, 'pid': None}]",None,0.0,publication,"{'code': 'fra/fre', 'label': 'French'}",None,"[{'subject': {'scheme': 'keyword', 'value': '[SDV.EE.SANT]Life Sciences [q-bio]/Ecology, environ...",E-health: feedback from patients and caregivers on daily life with digital technology,None,[Cette communication questionne les ressentis à propos de l'accent mis sur le numérique en santé...,2024-08-26,HAL CCSD,None,"[Ludovia #21 : quotidien & quotidienneté, https://hal.science/hal-04681617, Ludovia #21 : quotid...",None,[Centre de Recherche sur les Sociétés et Environnements en Méditerranées (CRESEM) ; Université d...,NaN,None,None,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,a9ac50f576aa::204d50c6df343ecf79129525b660989c,"[oai:hal.archives-ouvertes.fr:hal-04681617v1, 50|a9ac50f576aa::204d50c6df343ecf79129525b660989c]",None,2024-12-18T16:22:32.799,1.733521e+12,"{'citationImpact': {'citationCount': 0.0, 'influence': 2.8418670000000005e-09, 'popularity': 3.3...","[{'license': 'CC BY NC', 'type': 'Conference object', 'urls': ['https://hal.science/hal-04681617...",0.0,0.0
1,"[{'fullName': 'Barbara Zecchi', 'name': None, 'surname': None, 'rank': 1, 'pid': None}]",None,NaN,publication,"{'code': 'dan', 'label': 'Danish'}",None,"[{'subject': {'scheme': 'keyword', 'value': 'AZ20-999'}, 'provenance': {'provenance': 'Harvested...",An accented video way of thinking,None,[“An accented video way of thinking: Becoming videoessay” explores the videoessay as a conceivab...,2024-09-01,Aalborg University Open Publishing,None,"[Akademisk Kvarter, Iss 28 (2024)]",None,None,NaN,"{'code': 'c_abf2', 'label': 'OPEN', 'scheme': 'http://vocabularies.coar-repositories.org/documen...","{'name': 'Akademisk Kvarter', 'issnPrinted': '1904-0008', 'issnOnline': None, 'issnLinking': Non...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,doajarticles::3fe8781b55942dc2198f26ac79075f90,"[50|doajarticles::3fe8781b55942dc2198f26ac79075f90, oai:doaj.org/article:a9d232d863c64fe4acab9e7...",None,2024-12-18T16:22:23.757,1.733521e+12,"{'citationImpact': {'citationCount': 0.0, 'influence': 2.8418670000000005e-09, 'popularity': 3.3...","[{'alternateIdentifiers': [{'scheme': 'doi', 'value': '10.54337/academicquarter.i28.8852'}], 'ac...",NaN,NaN
2,"[{'fullName': 'James T. Gresh', 'name': 'James', 'surname': 'T. Gresh', 'rank': 1, 'pid': None},...",gold,0.0,publication,"{'code': 'und', 'label': 'Undetermined'}",None,None,Online Dating in The U.S. During This Politically Divided Time: Association Among Political Affi...,None,"[<jats:p>While there has been a surge in popularity of online dating, the current polarized poli...",2024-07-12,Institute for Promoting Research and Policy Development,None,[Crossref],None,None,NaN,"{'code': 'c_abf2', 'label': 'OPEN', 'scheme': 'http://vocabularies.coar-repositories.org/documen...","{'name': 'International Journal of Arts, Humanities &amp; Social Science', 'issnPrinted': '2693-...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,doi_________::017d7e694bb0cdee955f1c9bb46b0cd7,"[10.56734/ijahss.v5n7a3, 50|doiboost____|017d7e694bb0cdee955f1c9bb46b0cd7]","[{'scheme': 'doi', 'value': '10.56734/ijahss.v5n7a3'}]",2024-12-18T16:22:33.577,1.721003e+12,"{'citationImpact': {'citationCount': 0.0, 'influence': 2.8418670000000005e-09, 'popularity': 3.3...","[{'pids': [{'scheme': 'doi', 'value': '10.56734/ijahss.v5n7a3'}], 'accessRight': {'code': 'c_abf...",0.0,0.0


However, if you try to load the whole thing, even just the publications, the chances are slim.

If you try uncommenting and running the following lines, after some time, the kernel will die while trying and restart.

see how memory consume this process `htop` or similar

In [31]:
# files = sorted(glob.glob(f"{main_path}/publication/part*"))
# publications_df = pd.concat(pd.read_json(f, compression='gzip', lines=True) for f in files)

So, let's see how `Spark` can help us.

First thing first, let's create the Spark session.

In [3]:

spark = (
    SparkSession.builder
    .appName("My Spark Application")
    .master("local[8]")
    .config("spark.driver.memory", "3G")
    .config("spark.sql.shuffle.partitions", "200")
    .config("spark.sql.adaptive.enabled", "true")
    .config(
        "spark.jars.packages",
        "io.graphframes:graphframes-spark4_2.13:0.12.3"
    )
    .getOrCreate()
)

print("Spark Session created successfully!")
print("Spark version:", spark.version)

:: loading settings :: url = jar:file:/Users/miconis/IdeaProjects/Visual%20Studio%20Code/corso_openaire_unipi/.venv/lib/python3.12/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /Users/miconis/.ivy2.5.2/cache
The jars for the packages stored in: /Users/miconis/.ivy2.5.2/jars
io.graphframes#graphframes-spark4_2.13 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-fceff75c-33b2-4df1-9414-00e738b49dbf;1.0
	confs: [default]
	found io.graphframes#graphframes-spark4_2.13;0.12.3 in central
	found io.graphframes#graphframes-graphx-spark4_2.13;0.12.3 in central
	found org.apache.datasketches#datasketches-java;6.2.0 in local-m2-cache
	found org.apache.datasketches#datasketches-memory;3.0.2 in local-m2-cache
:: resolution report :: resolve 88ms :: artifacts dl 3ms
	:: modules in use:
	io.graphframes#graphframes-graphx-spark4_2.13;0.12.3 from central in [default]
	io.graphframes#graphframes-spar

Spark Session created successfully!
Spark version: 4.2.0


you can view the Spark UI [here](http://localhost:4040/jobs/)

In local mode, Spark runs inside a single JVM, and its heap is split into three regions: a fixed **300 MB of reserved memory**, **unified memory**, and **user memory**. Unified memory takes 60% of what remains after the reserved portion (`spark.memory.fraction = 0.6`); the other 40% is user memory, which holds your own data structures, UDF objects, and Spark's internal metadata.

Unified memory is shared by two consumers: **execution memory** (shuffles, joins, sorts, and aggregations) and **storage memory** (cached data and broadcast variables). The boundary between them is flexible. Execution can borrow any storage space that is currently free, but it cannot evict cached blocks from the protected storage region, which is 50% of unified memory by default (`spark.memory.storageFraction = 0.5`).

Execution memory is in turn shared among the tasks running concurrently. With `N` active tasks, each one is guaranteed between `1/(2N)` and `1/N` of the pool. A rough estimate of the per-task budget is:

$$
\text{Execution memory per task}
\approx
\frac{(\text{heap} - 300\ \text{MB}) \times 0.6}
{N}
$$

This is an upper bound, since it assumes that storage is empty and counts the whole unified pool. It also ignores off-heap memory, such as JVM overhead, I/O and network buffers, and the Python worker processes used by UDFs. On YARN, that overhead is configured with `spark.executor.memoryOverhead`.

## Extracting statistics with the mapped RDD

Now, let's extract some statistics using the mapped RDD.

In [5]:
df = spark.read.text(f"{main_path}/publication")

## Find the Number of Publications per Month

In [6]:
import json
date_regx = r"(\d{4})-(\d{2})-(\d{2})"

# Extract the publication month (MM) from a date formatted as YYYY-MM-DD.
# Return None when the date is missing or does not match the expected format.
def get_month(row)->str:
    date_str = row.get("publicationDate")
    match = re.search(date_regx, date_str)
    if match:
        return match.group(2)
    return None


mapped_rdd = df.rdd.map(lambda x : json.loads(x.value))

mapped_rdd.map(lambda x : (get_month(x), 1)).reduceByKey(lambda a, b: a + b).toDF(["month", "count"]).orderBy("month").toPandas()


,month,count
0,06,505784
1,07,607638
2,08,529296
3,09,587526
4,10,604980
5,11,522661
6,12,106694


## Find the Total Number of Authors per Month

The next cells compute the total number of authors per publication month. Each publication's `authors` list length is extracted with `get_num_authors`, then summed by month with `reduceByKey` on the RDD.

In [7]:

def get_num_authors(row) -> int:
    authors = row.get("authors",[])
    
    return len(authors) if authors is not None else 0

mapped_rdd.map(lambda x : (get_month(x), get_num_authors(x),)).reduceByKey(lambda a, b: a + b).toDF(["month", "total_authors"]).toPandas()

,month,total_authors
0,06,1839558
1,09,2512275
2,12,558481
3,08,2334273
4,07,2443651
5,11,2372130
6,10,2696789


## Find the Number of Publications per Typology

The next cell counts publications by typology. Each publication can have several `instances`, and each instance has a `type` (e.g. `Article`, `Preprint`). `get_typologies` collects the distinct types per publication, assigning `UNKNOWN` when a type is missing. The pairs are then summed with `reduceByKey` on the RDD.

In [8]:
def get_typologies(x)->list:
    instances = x.get("instances", [])
    return set([(i.get("type",'UNKNOWN'),1)  for i in instances]) if instances is not None else []


mapped_rdd.flatMap(lambda x: get_typologies(x)).reduceByKey(lambda a, b : a+ b).toDF(["typology", "count"]).toPandas()





,typology,count
0,Preprint,71969
1,Other literature type,362862
2,Newsletter,1
3,Data Management Plan,201
4,Book,37120
5,Master thesis,14079
6,External research report,493
7,Journal,12677
8,Report,18196
9,Patent,224


# Working with Dataframe

Now, let's read the datasets about OpenAIRE entitities.

In [9]:
publications = spark.read.json(f'{main_path}/publication')
datasets = spark.read.json(f'{main_path}/dataset')
software = spark.read.json(f'{main_path}/software')
others = spark.read.json(f'{main_path}/otherresearchproduct')
datasources = spark.read.json(f'{main_path}/datasource')
organizations = spark.read.json(f'{main_path}/organization')
projects = spark.read.json(f'{main_path}/project')
communities = spark.read.json(f'{main_path}/communities_infrastructures')
relations = spark.read.json(f'{main_path}/relation')

Ok, let's count the number of items for each typologies.

In [10]:
pd.DataFrame(
    [
    ("publication", publications.count()),
    ("datasets ",datasets.count()),
    ("software ", software.count()),
    ("other research products ", others.count()),
    ("datasources ",datasources.count()),
    ("organizations ",organizations.count()),
    ("communities",communities.count()),
    ("projects",projects.count()),
    ("relations",relations.count())
    ],
    columns=["Typology", "Count"],
)




,Typology,Count
0,publication,3464579
1,datasets,517869
2,software,23371
3,other research products,594379
4,datasources,57226
5,organizations,51382
6,communities,33
7,projects,58183
8,relations,24022968


## Generate a beautifull graph with plolty

In [11]:
import plotly.graph_objects as go
fig = go.Figure()
result = pd.DataFrame(
    [
    ("publication", publications.count()),
    ("datasets ",datasets.count()),
    ("software ", software.count()),
    ("other research products ", others.count()),
    ("datasources ",datasources.count()),
    ("organizations ",organizations.count()),
    ("communities",communities.count()),
    ("projects",projects.count()),
    ("relations",relations.count())
    ],
    columns=["Typology", "Count"],
)

fig.add_bar(x=result["Typology"], y=result["Count"])
fig.show()


## Visualise sample data structures

Let's show some data now. 

### Publication
For example, a generic publication (link to documentation: https://graph.openaire.eu/docs/data-model/entities/research-product)

In [12]:
pretty_print(json.loads(publications.toJSON().first()), expanded=False)

/Users/miconis/IdeaProjects/Visual Studio Code/corso_openaire_unipi/.venv/lib/python3.12/site-packages/pyspark/sql/classic/dataframe.py:178: UserWarning: The return type of DataFrame.toJSON will be changed from RDD to DataFrame in future releases.
  warnings.warn(
26/11/23 08:36:09 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


<IPython.core.display.JSON object>

### **Task:** find publications with a keyword in the title, e.g., `covid`


In [13]:
publications.where("mainTitle LIKE '%covid%'").select("mainTitle",F.explode("Instances.type").alias("InstanceType") ).limit(30).toPandas()

,mainTitle,InstanceType
0,Kendala guru dalam meningkatkan kreativitas belajar siswa pada masa pandemi covid-19 di kelas re...,Article
1,Wuhan market the most likely source of covid-19 outbreak,Article
2,O uso de laboratório virtual no ensino de Ciências da Natureza diante da conjuntura da covid-19:...,Article
3,"Parliamentarians and the covid-19 pandemic: insights from an executive-dominated, constituency-o...",Article
4,O efeito da quimioterapia citotóxica e do bloqueio androgênico na covid-19 e o impacto da pandem...,Doctoral thesis
5,A PANDEMIA E O FEMININO: Uma análise sobre os impactos do cenário pós covid-19 para as mulheres,Article
6,Determinación de la potencia mecánica en pacientes con síndrome de distrés respiratorio agudo po...,Article
7,Factors involving incomplete vaccination in children under 2 years old in the covid-19 pandemic,Article
8,Os impactos da covid-19 no sistema cardiovascular do idoso,Article
9,The impact of the covid-19 epidemic on tourism in Portugal and the mechanisms adopted during and...,Article


In [14]:

publications.selectExpr("month(to_date(publicationDate)) AS month").groupBy("month").count().toPandas()

,month,count
0,12,106694
1,6,505784
2,9,587526
3,8,529296
4,7,607638
5,10,604980
6,11,522661


In [15]:
import plotly.graph_objects as go
fig = go.Figure()
result = publications.selectExpr("month(to_date(publicationDate)) AS month").groupBy("month").count().toPandas()
fig.add_bar(x=result["month"], y=result["count"])
fig.show()

### **Task:** count publications by language

In [16]:
import plotly.graph_objects as go
fig = go.Figure()
result = publications.selectExpr("language.code as languageCode").where("languageCode != 'und'").groupBy("languageCode").count().orderBy(F.desc("count")).limit(25).toPandas()
fig.add_bar(x=result["languageCode"], y=result["count"])
fig.show()


## Relation queries

### **Task:** count and sort by relation typologies

In [17]:
result = relations.groupBy("relType.name").count().orderBy(F.desc("count")).limit(20).toPandas()
fig = go.Figure()
fig.add_bar(x=result["name"], y=result["count"])
fig.show()


## Advanced queries

### **Task:** count and sort publications by citations

In [18]:
relations.where("relType.name = 'Cites'").groupBy("target").count().orderBy(F.desc("count")).show(truncate=False)

+----------------------------------------------+-----+
|target                                        |count|
+----------------------------------------------+-----+
|doi_________::0ac3d36dd94066ade5cc874c0e02a61c|1556 |
|doi_________::38d83a048882b01389f2332bac93bb05|1547 |
|doi_________::01c72a8901b50eb8875897c9094c97a4|1540 |
|doi_________::ab98cd984ef7bdb10a9da297ec2d483d|1535 |
|doi_________::771d7ea290e39b6803621767c9a9bc4d|1529 |
|doi_________::98dba3f964364b03bfad0ff91b7288ce|1353 |
|doi_________::0cd7be367fed80650281b537d1a48443|943  |
|doi_________::1570b298bbd2c41eb36d99a10a356390|942  |
|doi_________::cf24aec00aca4fef461436e7b86e957e|940  |
|doi_________::6e315dcc608e905282aff925b564b83a|940  |
|doi_________::1cbe2c2f932dd530451bdfbe01a3fe84|939  |
|doi_________::83475c8870b4c84e6f804ea0a40a0297|938  |
|doi_________::de6aeeb122a6fd4602318f49e899c635|938  |
|doi_________::c02282500cca53f9b6afc2bcae3c57c8|937  |
|doi_________::419f7216467ef29efe2f755b3ecb5fe5|936  |
|doi______

### **Task:** count and sort publications by citations and show also the first occurrence of pid

In [19]:
(publications
    .join(relations, relations["target"]==publications['id']).where("relType.name = 'Cites'")
    .selectExpr("id", "pids[0] as mainPid", "relType.name")
    .groupBy("id", "mainPid.value").count().orderBy(F.desc("count")).limit(20).toPandas()
)

,id,value,count
0,doi_________::0ac3d36dd94066ade5cc874c0e02a61c,10.38124/ijisrt/ijisrt24may2042,1556
1,doi_________::38d83a048882b01389f2332bac93bb05,10.38124/ijisrt/ijisrt24jun909,1547
2,doi_________::01c72a8901b50eb8875897c9094c97a4,10.38124/ijisrt/ijisrt24apr2181,1540
3,doi_________::ab98cd984ef7bdb10a9da297ec2d483d,10.38124/ijisrt/ijisrt24jun949,1535
4,doi_________::771d7ea290e39b6803621767c9a9bc4d,10.38124/ijisrt/ijisrt24may2437,1529
5,doi_________::98dba3f964364b03bfad0ff91b7288ce,10.38124/ijisrt/ijisrt24jun1035,1353
6,doi_________::0cd7be367fed80650281b537d1a48443,10.38124/ijisrt/ijisrt24jun654,943
7,doi_________::1570b298bbd2c41eb36d99a10a356390,10.38124/ijisrt/ijisrt24jul073,942
8,doi_________::cf24aec00aca4fef461436e7b86e957e,10.38124/ijisrt/ijisrt24jul244,940
9,doi_________::6e315dcc608e905282aff925b564b83a,10.38124/ijisrt/ijisrt24may2382,940


### **Task:** show the journals with the highest number of publications

In [20]:
result = (
publications.selectExpr("container.name as journalName", "id")
    .where("journalName is not null")
    .groupBy("journalName")
    .count()
    .orderBy(F.desc("count"))
    .toPandas().head(10)

)

fig = go.Figure()
fig.add_pie(
    labels=result["journalName"],
    values=result["count"],
    title="Top journals",
)
fig.show()

### **Task:** count OA statuses of all products

In [21]:
cols = ["id", "bestAccessRight", "publicationDate"]

products = (
    publications.select(*cols)
    .union(datasets.select(*cols))
    .union(software.select(*cols))
    .union(others.select(*cols))
    .distinct()
)

result = (
    products
    .where("bestAccessRight is not null")
    .selectExpr("bestAccessRight.label as accessright",
        "substring(publicationDate, 1, 4) as year"
    )
    .groupBy("accessright", "year")
    .count()
    .orderBy(F.desc("count"))
    .limit(20)
    .toPandas()
)

fig = go.Figure()
for accessright, group in result.groupby("accessright"):
    fig.add_bar(x=group["year"], y=group["count"], name=accessright)
fig.update_layout(
    barmode="group",
    xaxis_title="Year",
    yaxis_title="Count",
    title="Products by access right and year",
)
fig.show()

### **Task:** deduplication of Author

First thing first, let's create a new dataframe for authors.

In [44]:
from pyspark.sql.functions import explode, col, udf
from pyspark.sql.types import StringType
import hashlib

authors = (
    publications.selectExpr("id as resultId", "explode(authors) as author")
    .union(datasets.selectExpr("id as resultId", "explode(authors) as author"))
    .union(software.selectExpr("id as resultId", "explode(authors) as author"))
    .union(others.selectExpr("id as resultId", "explode(authors) as author"))
).select("resultId", "author.*")

@udf(StringType())
def generate_author_id(result_id, fullname):
    value = f"{result_id}::{fullname}"
    return "author::" + hashlib.md5(value.encode("utf-8")).hexdigest()

authors = authors.withColumn("authorId", generate_author_id(col("resultId"), col("fullName")))

authors.printSchema()

authors.write.mode("overwrite").save(f'{main_path}/authors')


root
 |-- resultId: string (nullable = true)
 |-- fullName: string (nullable = true)
 |-- name: string (nullable = true)
 |-- pid: struct (nullable = true)
 |    |-- id: struct (nullable = true)
 |    |    |-- scheme: string (nullable = true)
 |    |    |-- value: string (nullable = true)
 |    |-- provenance: struct (nullable = true)
 |    |    |-- provenance: string (nullable = true)
 |    |    |-- trust: string (nullable = true)
 |-- rank: long (nullable = true)
 |-- surname: string (nullable = true)
 |-- authorId: string (nullable = true)



#### Naive Deduplication

In [4]:
authors = spark.read.load(f'{main_path}/authors')
authors.count()

16164816

In [5]:
authors.selectExpr("lower(fullName) as fullName").groupBy("fullName").count().orderBy(F.desc("count")).show()

+--------------------+-----+
|            fullName|count|
+--------------------+-----+
|       paul maeyaert|36546|
|  tokuzawa, tokihiko|33530|
|     trimmel, gregor|28017|
|     schober, isabel|16017|
|      overmann, jörg|16016|
|   reimer, lorenz c.|16015|
|    witte, julius f.|16012|
|       lissin, artur|16012|
|         bunk, boyke|16011|
|     podstawka, adam|16011|
|        lüken, helko|16011|
| kobayashi, masahiro|12592|
|    masuzaki, suguru|11397|
|      dominik ukolov| 9128|
|research center d...| 9127|
|       tanaka, kenji| 8199|
|        overmann, j.| 7920|
|        reimer, l.c.| 7919|
|         koblitz, j.| 7919|
|       podstawka, a.| 7919|
+--------------------+-----+
only showing top 20 rows


In [6]:
authors.selectExpr("lower(fullName) as fullName").groupBy("fullName").count().count()

7522822

Drawbacks of this naive disambiguation:

1. Author names with typos will never be identified as duplicates
2. Homonyms will be identified as duplicates even when they are not

#### Smart deduplication

1. Group authors into clusters of potentially equivalent authors (Blocking)
2. Pair-wise comparisons between authors in the same block (Matching)
3. Transitive closure to compute groups of equivalent authors (Deduplication)

##### Blocking

We need a function to emit a clustering key from an author. Such a key will be used to group authors into blocks.

In [34]:
import re

def is_not_blank(value):
    return value is not None and bool(str(value).strip())

def generate_key(author):
    if is_not_blank(author['surname']) and is_not_blank(author['name']):
        return str(author['surname'] + author['name'][0]).lower()   # Last Name First Initial
    return re.sub(r'[\W_ ]+', '', author['fullName']).lower()   # remove punctuations from the fullname and use it as a key


Now we can emit a clustering key for each author.

In [35]:
authors_for_dedup = authors.rdd.map(lambda author: (generate_key(author), author))

for row in authors_for_dedup.take(20):
    print(row)

('bonoe', Row(resultId='a9ac50f576aa::13f44438c33f9061574b27b1fdd54d24', fullName='de Bono, Emanuele', name='Emanuele', pid=None, rank=1, surname='Bono', authorId='author::fd00f599f075d920710d1a075b710731'))
('fernandesa', Row(resultId='a9ac50f576aa::13f44438c33f9061574b27b1fdd54d24', fullName='Fernandes, Arthur, Diniz', name='Arthur', pid=None, rank=2, surname='Fernandes', authorId='author::0d7a99bb66bb827aea83a566252e9ce9'))
('petroneg', Row(resultId='a9ac50f576aa::13f44438c33f9061574b27b1fdd54d24', fullName='Petrone, Giuseppe', name='Giuseppe', pid=None, rank=3, surname='Petrone', authorId='author::0068ea81a5a9937087361f6f05fd3c6d'))
('ouissem', Row(resultId='a9ac50f576aa::13f44438c33f9061574b27b1fdd54d24', fullName='Ouisse, Morvan', name='Morvan', pid=None, rank=4, surname='Ouisse', authorId='author::c392551e45e5871dfe1844e46031a9d6'))
('telolir', Row(resultId='a9ac50f576aa::13f44438c33f9061574b27b1fdd54d24', fullName='Teloli, Rafael', name='Rafael', pid=None, rank=5, surname='Telo

And create blocks based on the clustering key.

In [33]:
blocks = authors_for_dedup.groupByKey()

for block in blocks.take(20):
    print(block)

('soliva domínguezr', <pyspark.resultiterable.ResultIterable object at 0x127698170>)
('ghamrye', <pyspark.resultiterable.ResultIterable object at 0x12769a180>)
('doburd', <pyspark.resultiterable.ResultIterable object at 0x12769b020>)
('stelm', <pyspark.resultiterable.ResultIterable object at 0x12769b1a0>)
('couletf', <pyspark.resultiterable.ResultIterable object at 0x131b4b170>)
('shamilovd', <pyspark.resultiterable.ResultIterable object at 0x131b4a450>)
('fultons', <pyspark.resultiterable.ResultIterable object at 0x131b4b200>)
('giardinos', <pyspark.resultiterable.ResultIterable object at 0x131b4af60>)
('youbi', <pyspark.resultiterable.ResultIterable object at 0x1304c3620>)
('fanerr', <pyspark.resultiterable.ResultIterable object at 0x1304c1040>)
('bradsher', <pyspark.resultiterable.ResultIterable object at 0x1309d9220>)
('elalamid', <pyspark.resultiterable.ResultIterable object at 0x1309d9310>)
('landsr', <pyspark.resultiterable.ResultIterable object at 0x1309d9790>)
('enzoiuliano',

##### Matching

We need a function to perform a pair-wise comparison.

In [37]:
import unicodedata

def pair_wise_comparison(a, b):
    # smart comparison to save computation time

    if a['resultId'] == b['resultId']:
        return False  # if authors come from the same result, they are different

    # pid comparison
    if a['pid'] is not None and b['pid'] is not None:  # if both authors have pids
        if a['pid']['id']['value'] == b['pid']['id']['value']:
            return True  # when the pid is the same
        else:
            return False # when the pid is different

    if a['name'] is not None and b['name'] is not None and a['surname'] is not None and b['surname'] is not None:   # if both authors have name and surname
        if normalize(a['surname']) == normalize(b['surname']):
            if normalize(a['name']) == normalize(b['name']):
                return True   # if name and surname are exactly the same
    
    return False

# normalization function
def normalize(value):
    value = unicodedata.normalize("NFKD", value)
    value = "".join(c for c in value if not unicodedata.combining(c))
    return re.sub(r"[^a-z0-9]", "", value.lower())

And now a function to perform all the pair-wise comparisons within a block.

In [43]:
from itertools import combinations

def process_block(block):
    clustering_key, authors_iterator = block
    relations = []

    for a, b in combinations(authors_iterator, 2):  # compute all the combinations in the group

        if pair_wise_comparison(a, b):   # if the comparison succeeded, draw a similarity relations
            relations.append({"source": a['authorId'], "target": b['authorId']})

    return pd.DataFrame(relations, columns=["source", "target"])

def process_block_with_sliding_window(block, window_size=10):
    clustering_key, authors_iterator = block

    authors = list(authors_iterator)

    if len(authors) > 100:  # ignore large groups (unreliable)
        return []

    authors.sort(key=lambda author: author['fullName'])     # sort authors by their name

    relations = []

    for index, author_a in enumerate(authors):
        # compare only with the authors in the same window
        window = authors[index + 1:index + 1 + window_size]

        for author_b in window:
            if pair_wise_comparison(author_a, author_b):
                relations.append({"source": author_a["authorId"],"target": author_b["authorId"],})

    return relations


Ok, now let's draw similarity relationships.

In [44]:
from pyspark.sql.types import StructField

simrels_rdd = blocks.flatMap(process_block_with_sliding_window)

simrels = spark.createDataFrame(simrels_rdd, schema=StructType([StructField("source", StringType(), False), StructField("target", StringType(), False)]))

simrels.write.mode("overwrite").save(f'{main_path}/simrels')
simrels.count()

13857611

#### Deduplication

We need a library to perform the transitive closure for us.

In [45]:
from graphframes import GraphFrame

authors = spark.read.load(f'{main_path}/authors')
simrels = spark.read.load(f'{main_path}/simrels')

# preparation of the data structures to perform transitive closure
vertices = authors.selectExpr("authorId as id").where("id is not null").distinct()
edges = simrels.selectExpr("source as src", "target as dst").where("src is not null and dst is not null").distinct()

g = GraphFrame(vertices, edges)

result = g.connectedComponents(
    algorithm="two_phase",  # graphframes strategy to perform the transitive closure
    broadcastThreshold=-1,  # disable automatic broadcasting
    use_local_checkpoints=True
)

deduped_authors = (
    authors
    .join(
        result.select(col("id").alias("authorId"), col("component").alias("authorDedupId")),
        on="authorId",
        how="left"
    )
)

deduped_authors.write.mode("overwrite").save(f'{main_path}/authors_deduped')


26/11/23 14:23:10 WARN TwoPhase$: Returned DataFrame is persistent and materialized!


Let's check the results.

In [ ]:
deduped_authors = spark.read.load(f'{main_path}/authors_deduped')

deduped_authors.groupBy("authorDedupId").agg(
    F.count("*").alias("count"), F.sort_array(F.collect_set("fullName")).alias("names")
).orderBy(F.desc("count")).show(truncate=False)

deduped_authors.select("authorDedupId").distinct().count()

+-------------+-----+----------------------------------------------------------------------------------------------------+
|authorDedupId|count|names                                                                                               |
+-------------+-----+----------------------------------------------------------------------------------------------------+
|32704        |100  |[Afedulidis, O, Afedulidis, O., O. Afedulidis]                                                      |
|8589989368   |100  |[sdfdsf, sdfdf]                                                                                     |
|3716         |99   |[Arnaud Henrard, Henrard, Arnaud]                                                                   |
|8589981930   |99   |[Coulis, Mathieu, MATHIEU COULIS, Mathieu Coulis]                                                   |
|72791        |99   |[DMITRY TELNOV, Dmitry Telnov, Telnov, Dmitry]                                                      |
|13566        |9

12089981